# 12m — Teste do SVM multiclasse (padrão e ajustado), uma única vez cada, com o `Portmap` à parte

Este notebook avalia no **conjunto de teste** os dois **modelos finais (refit) do SVM multiclasse** do notebook 11m: o **padrão** (C = 1, gamma = 1× o de referência) e o **escolhido pela busca** (C = 100, gamma = 4×), **nas mesmas 7.216.487 linhas** em que o RF foi avaliado (notebooks 06m e 08m), e compara os quatro modelos. **Nada é treinado nem ajustado aqui**, e o teste é usado **uma única vez para cada modelo**.

## O que é igual ao 06m e ao 08m (RF)
| Item | Como |
|---|---|
| **Dados** | `prep_vw_rf_multi_teste_sem_sobrep`: as 7.216.487 linhas do outro dia, sem sobreposição com o treino e só das 7 classes. **Mistura natural** (o `Syn` é ~68%) |
| **Previsão** | A classe de **maior escore** (decisão um-contra-resto do SVM) |
| **Métricas** | Recall, precisão e F1 por classe (com intervalo de 95% **otimista**), acurácia balanceada, F1 macro (também sem o `UDPLag`), MCC, ROC-AUC um-contra-resto, matriz de confusão 7×7, validação × teste |
| **`Portmap`** | Classe que só existe no teste: **fora das métricas principais**, com seção própria (para qual classe é empurrada e com que confiança) |
| **Comparação** | Os quatro modelos lado a lado (RF padrão, RF ajustado, SVM padrão, SVM ajustado), com a **faixa das 5 sementes** do RF (notebook 09m), o desempenho **ataque × BENIGN** e a comparação **linha a linha** |

## O que é diferente: a previsão só dos vetores únicos (resultado idêntico)
O SVM com ~78 mil vetores de suporte prevê **~78 fluxos/s** (medido no 11m): as 7,2 milhões de linhas levariam **~26 h por modelo**. Mas o teste tem **muitas linhas com exatamente as mesmas 44 features**, e o SVM é uma função determinística dessas 44 features: **o mesmo vetor sempre dá exatamente a mesma previsão**. Por isso:
1. O notebook lê o teste **uma vez** e identifica os **vetores distintos** (comparação **exata**, byte a byte, por uma chave de 128 bits; nada de arredondar).
2. **Prevê cada vetor distinto uma única vez.**
3. **Espalha** o resultado para **todas as linhas** com aquele vetor. As métricas são calculadas sobre as **7.216.487 linhas**, cada uma com o seu rótulo real (linhas com os mesmos vetores e rótulos diferentes, os conflitos, recebem a mesma previsão e cada uma conta com o seu rótulo).
4. **Prova da equivalência:** uma amostra de **3.000 linhas do teste** (com repetidas e conflitantes) é prevista **diretamente** pelo modelo e comparada com o resultado espalhado. Elas precisam ser **idênticas**.

O RF não precisou do atalho (previu tudo em ~1 min), e o resultado dele é o mesmo de qualquer forma.

## Paralelismo: só com memória suficiente
A previsão dos vetores distintos é dividida em tarefas independentes (um arquivo de vetores cada), e cada tarefa roda **num processo à parte**. O notebook roda **várias ao mesmo tempo, mas só se houver memória livre** (a mesma regra do 11m: memória por processo × 1,3 mais 1 GiB de reserva, partidas espaçadas, tarefa interrompida e refeita se a memória ficar crítica, nunca trava com um processo só). Cada tarefa é salva ao terminar: **se interromper, rode de novo e ele retoma**.

## Garantias (verificadas por código)
1. **Nenhum modelo é treinado.** O `fit` do SVM e do RF dá erro se for chamado.
2. **Nenhum arquivo existente é sobrescrito ou apagado**, nas pastas do SVM (11m), do RF (07m/08m e 05m/06m). Tudo é criado em `teste_ajustado/`, `teste_padrao/` e arquivos novos; cada arquivo só é criado se ainda não existir.
3. **O servidor só é lido** (só `SELECT`).
4. **Conferência final:** o tamanho e a data de **todos os arquivos que já existiam** nas pastas do SVM e do RF são conferidos antes e depois.
5. **Uso único do teste, por modelo:** o `metricas_teste.json` de cada modelo é gravado **por último**. Se já existir, o modelo **não é reavaliado** (o notebook só recarrega o resultado), de modo que dá para rodar o notebook de novo sem gastar o teste.

## O que é criado (na pasta `svm_busca_<data>`)
`teste_ajustado/` e `teste_padrao/` (cada um com `predicoes_teste.npz`, `metricas_teste.csv/json`, `metricas_por_classe_teste.csv`, `matriz_confusao_teste.csv`, `validacao_vs_teste.csv`, `portmap_teste.csv`, `custo_inferencia_teste.json`, o gráfico), `mapa_teste/` (o mapa linha → vetor distinto), `vetores_unicos/` e `previsoes/` (para retomar), `comparacao_svm_vs_rf_*.csv` e uma linha em `registro_testes.csv`.

## Antes de rodar
- O 11m precisa ter terminado (pasta com `FINALIZADO`), e o 07m, o 08m, o 06m e o 09m também (para a comparação). O `.env` precisa apontar para o servidor novo.
- **Disco:** ~0,5 a 1,5 GB (vetores distintos em `float64` + escores).
- **Memória:** ~0,3 GiB do mapa e ~0,2 GiB dos escores no fim, mais ~0,5 GiB por processo de previsão. Feche os outros notebooks.
- **Tempo (estimativa, sem medir):** ~7 min para ler o teste e achar os vetores distintos; depois, **a previsão**: ~2,8 h por modelo se ~11% das linhas forem distintas (~5 h se forem 20%), com 1 processo; com 2 processos, ~1,5 h por modelo. O notebook **conta os vetores distintos e mostra o tempo estimado** antes de prever; se for muito maior, você pode interromper sem perder nada.


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas
O pré-processamento, a leitura do teste em CSV, as métricas multiclasse para arrays grandes, o **escalonador de tarefas condicionado à memória** e as **proteções** (treino, sobrescrita e remoção bloqueados).

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# ---------- métricas multiclasse (a partir da matriz de confusão; aceitam pesos por linha) ----------
def metricas_multi(y, proba, w=None):
    y = np.asarray(y, dtype=int); proba = np.asarray(proba, dtype=float); K_ = proba.shape[1]; pred = proba.argmax(axis=1); w_ = None if w is None else np.asarray(w, dtype=float)
    cm = np.zeros((K_, K_)); np.add.at(cm, (y, pred), 1.0 if w_ is None else w_)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)                  # total, linhas por classe verdadeira, previstas por classe, acertos
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k).astype(int)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, proba[:, k], sample_weight=w_)
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t}

# autoteste das ferramentas (não toca no servidor): confere as métricas multiclasse contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc, precision_score as _prec
_r = np.random.default_rng(0); _y = _r.integers(0, 4, 400); _P = _r.dirichlet(np.ones(4), 400); _P[np.arange(400), _y] += 0.35; _P /= _P.sum(axis=1, keepdims=True); _w = _r.uniform(0.5, 3, 400); _pr = _P.argmax(axis=1)
for _ww in (None, _w):
    _e, _a = metricas_multi(_y, _P, _ww)
    assert abs(_e["acuracia"] - _acc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr, sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["f1_macro"] - _f1(_y, _pr, average="macro", sample_weight=_ww)) < 1e-12 and abs(_e["f1_ponderado"] - _f1(_y, _pr, average="weighted", sample_weight=_ww)) < 1e-12, _e
    assert abs(_e["mcc"] - _mcc(_y, _pr, sample_weight=_ww)) < 1e-12 and abs(_e["precisao_macro"] - _prec(_y, _pr, average="macro", sample_weight=_ww, zero_division=0)) < 1e-12, _e
    assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P, multi_class="ovr", average="macro", sample_weight=_ww)) < 1e-12, _e
assert abs(metricas_multi(np.array([0, 1, 2]), np.eye(3))[0]["mcc"] - 1.0) < 1e-12
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- escalonador de tarefas condicionado à memória (o mesmo do 11m) ----------
import collections
def ram_livre_gib():
    s = globals().get("RAM_LIVRE_SIMULADA")                                                                     # só para testes
    return float(s() if callable(s) else s) if s is not None else (psutil.virtual_memory().available / 2 ** 30 if psutil else float("inf"))

def processos_seguros(desejado, ram_por_proc_gib):
    # Quantos processos simultâneos cabem AGORA: cada um precisa de ram_por_proc × 1,3, e a reserva nunca é tocada. Sempre devolve pelo menos 1 (nunca trava), mas avisa se nem 1 cabe folgado.
    livre = ram_livre_gib() - RAM_RESERVA_GIB
    if not np.isfinite(livre): return int(desejado)
    return int(max(1, min(desejado, livre // (ram_por_proc_gib * 1.3))))

def rodar_tarefas(pendentes, lancar, coletar, barra, limite_s, k_max, ram_proc, rotulo, parar_apos=None):
    # Escalonador de treinos independentes, UM PROCESSO NOVO por treino. Só inicia mais um treino se houver memória livre para ele (ram_proc × 1,3 + reserva) e se o anterior já ocupou a sua memória;
    # com um treino só rodando, sempre inicia o próximo (nunca trava). Se a memória ficar crítica com vários treinos, o mais novo é interrompido e volta para a fila. Treino acima do limite de tempo é encerrado.
    espera_desde = None; fila, rodando, feitos, ultima, stats = collections.deque(pendentes), [], 0, 0.0, {"max_simultaneos": 0, "interrompidos_por_memoria": 0, "ram_proc_final": ram_proc}
    try:
        while fila or rodando:
            while fila and len(rodando) < k_max:
                livre = ram_livre_gib(); agora = time.perf_counter()
                if rodando and (agora - ultima < INTERVALO_PARTIDAS_S or livre - RAM_RESERVA_GIB < ram_proc * 1.3): break
                if not rodando:
                    if livre < RAM_MINIMA_UM_TREINO_GIB:                                  # com nada rodando, espera a memória liberar (até 10 min); depois, para com uma mensagem clara
                        espera_desde = espera_desde or agora
                        if agora - espera_desde > 600: raise RuntimeError(f"Falta memória livre: um treino precisa de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB livres e há {livre:.1f} GiB. Feche o Meet, o Edge e os outros notebooks (ou defina RAM_MINIMA_UM_TREINO_GIB mais baixo, por sua conta e risco).")
                        barra._extra = f"aguardando memória livre ({livre:.1f} GiB; preciso de pelo menos {RAM_MINIMA_UM_TREINO_GIB:.1f} GiB)..."; barra._mostrar(); time.sleep(2.0); break
                    espera_desde = None
                t = fila.popleft(); ex, fut = lancar(t); rodando.append({"t": t, "ex": ex, "fut": fut, "t0": time.perf_counter(), "k": len(rodando) + 1}); ultima = time.perf_counter(); stats["max_simultaneos"] = max(stats["max_simultaneos"], len(rodando))
            for r in list(rodando):
                dec = time.perf_counter() - r["t0"]
                if r["fut"].done():
                    res = r["fut"].result(); r["ex"].shutdown(wait=True); rodando.remove(r); feitos += 1; ram_proc = max(ram_proc, 1.15 * res[0]["pico_ram_mib"] / 1024); stats["ram_proc_final"] = ram_proc; coletar(r["t"], res, r["k"])
                    if parar_apos and feitos >= parar_apos: raise RuntimeError("parada de teste (PARAR_APOS_FITS)")
                elif dec > limite_s: r["ex"].shutdown(wait=True, kill_workers=True); rodando.remove(r); feitos += 1; coletar(r["t"], None, r["k"])
            if len(rodando) > 1 and ram_livre_gib() < RAM_CRITICA_GIB:
                r = rodando.pop(); r["ex"].shutdown(wait=True, kill_workers=True); fila.appendleft(r["t"]); k_max = max(1, k_max - 1); stats["interrompidos_por_memoria"] += 1
                print(f"  ⚠ memória livre crítica ({ram_livre_gib():.2f} GiB): um treino foi interrompido e volta para a fila; o limite de treinos simultâneos agora é {k_max}")
            barra._extra = f"{rotulo} | {len(rodando)} treino(s) rodando (até {k_max}) | memória livre {ram_livre_gib():.1f} GiB | " + (f"mais antigo há {fmt_tempo(max(time.perf_counter() - r['t0'] for r in rodando))}" if rodando else "")
            barra._mostrar(); time.sleep(0.5)
    except BaseException:
        for r in rodando: r["ex"].shutdown(wait=True, kill_workers=True)
        raise
    return stats

# ---------- proteções: o que este notebook NUNCA faz (treinar, sobrescrever, apagar) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

TREINOS_BLOQUEADOS, RAIZES_TRAVADAS, CRIADOS, MANTIDOS = [], [], [], []
def instalar_travas(pasta, permitir_regravar=()):
    # (1) o treino do SVM e do RF é BLOQUEADO; (2) bloqueia gravação que SOBRESCREVA arquivo existente dentro das pastas protegidas, e qualquer remoção de arquivo dentro delas.
    # Pode ser chamada para várias pastas; só arquivos temporários (.tmp) e acréscimos (modo "a") podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    from sklearn.svm import SVC as _SVC
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    RAIZES_TRAVADAS.append(str(Path(pasta).resolve())); perm = set(permitir_regravar)
    if getattr(_SVC.fit, "_bloqueado", False): return
    def dentro(alvo):
        try: s_ = str(Path(alvo).resolve())
        except Exception: return False
        return any(s_.startswith(r) for r in RAIZES_TRAVADAS)
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file"); embrulha(_np, "save", 0, "gravar NPY", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}'. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}'. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u
    for _cls in (_RF, _SVC):
        def _mk(orig):
            def _fit_bloqueado(self, *a, **k):
                TREINOS_BLOQUEADOS.append(1); raise RuntimeError("BLOQUEADO: treinar um modelo não é permitido neste notebook (ele só carrega os modelos já treinados).")
            _fit_bloqueado._bloqueado = True; return _fit_bloqueado
        _cls.fit = _mk(_cls.fit)

def gravar(caminho, escrever):
    # Cria o arquivo SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = Path(caminho); rotulo = f"{caminho.parent.name}/{caminho.name}"
    if caminho.exists(): MANTIDOS.append(rotulo); return False
    caminho.parent.mkdir(parents=True, exist_ok=True); escrever(caminho); CRIADOS.append(rotulo); return True
def gravar_csv(df, caminho, **kw): return gravar(caminho, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, caminho): return gravar(caminho, lambda p: salvar_json(obj, p))
def tmp_novo(caminho): return Path(caminho).with_name(Path(caminho).stem + ".tmp" + Path(caminho).suffix)

print("Ferramentas do teste do SVM prontas")


Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK
Ferramentas do teste do SVM prontas


## 1. Parâmetros e carga
Localiza o SVM finalizado (11m) e os resultados do RF (06m e 08m), **ativa as proteções**, tira a fotografia dos arquivos existentes e confere cada modelo gravado (hiperparâmetros, classes, atributos, vetores de suporte).

In [6]:
# ---------- 1.1 Parâmetros e carga ----------
import joblib, hashlib, collections
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)                    # NÃO mude no meio de uma execução (o mapa do teste usa esse tamanho)
PERMITIR_REPETIR = globals().get("PERMITIR_REPETIR", False)           # True = recalcula as métricas de um modelo já avaliado (os arquivos existentes continuam intactos)
SEMENTE = globals().get("SEMENTE", 42); N_AMOSTRA_EQUIV = globals().get("N_AMOSTRA_EQUIV", 3000)       # linhas do teste previstas DIRETAMENTE para provar a equivalência
# --- paralelismo da previsão, condicionado à memória (a mesma regra do 11m) ---
PROCESSOS_PREVISAO_MAX = globals().get("PROCESSOS_PREVISAO_MAX", 4); RAM_POR_PROCESSO_GIB = globals().get("RAM_POR_PROCESSO_GIB", 0.6)
RAM_RESERVA_GIB = globals().get("RAM_RESERVA_GIB", 1.0); RAM_MINIMA_UM_TREINO_GIB = globals().get("RAM_MINIMA_UM_TREINO_GIB", 0.5); RAM_CRITICA_GIB = globals().get("RAM_CRITICA_GIB", 0.4)
INTERVALO_PARTIDAS_S = globals().get("INTERVALO_PARTIDAS_S", 10); LIMITE_TAREFA_S = globals().get("LIMITE_TAREFA_S", 4 * 3600); PARAR_APOS_TAREFAS = globals().get("PARAR_APOS_TAREFAS", None)       # só para testes
nome_cfg = lambda C, f: f"C{C:g}_g{f:g}"
BASE_ART = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos"; PASTA_BASE = BASE_ART / "svm_multiclasse_busca"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("svm_busca_*") if (p / "FINALIZADO").exists() and (p / "modelo_final_refit.joblib").exists())
    assert runs, "nenhuma pasta svm_busca_* FINALIZADA com modelo_final_refit.joblib em " + str(PASTA_BASE) + ": rode antes o notebook 11m (ou defina PASTA_RUN)"
    if len(runs) > 1: print("⚠ há mais de uma busca finalizada; usando a mais recente. Para escolher outra, defina PASTA_RUN:", [p.name for p in runs])
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN); assert (PASTA_RUN / "FINALIZADO").exists(), f"{PASTA_RUN.name} não está FINALIZADA: rode o 11m até o fim"
def ultima_com(base, padrao, arquivos):
    ps = sorted(p for p in (BASE_ART / base).glob(padrao) if all((p / a).exists() for a in arquivos)); return ps[-1] if ps else None
PASTA_RF_AJ = Path(globals().get("PASTA_RF_AJ") or ultima_com("rf_multiclasse_busca", "rfm_busca_*", ("metricas_teste.json", "predicoes_teste.npz")) or ".")
PASTA_RF_PAD = Path(globals().get("PASTA_RF_PAD") or ultima_com("rf_multiclasse", "rfm_*", ("metricas_teste.json", "predicoes_teste.npz")) or ".")
assert (PASTA_RF_AJ / "metricas_teste.json").exists() and (PASTA_RF_PAD / "metricas_teste.json").exists(), "faltam os resultados do teste do RF (08m e 06m): defina PASTA_RF_AJ e PASTA_RF_PAD"
VIEW_TESTE, TAB_TESTE, TAB_CLASSES, V_VALID = "prep_vw_rf_multi_teste_sem_sobrep", "prep_final_teste", "prep_classes_rf_multi", "prep_vw_bal_multi_validacao"
assert existe(VIEW_TESTE) and existe(TAB_CLASSES) and existe(V_VALID), f"faltam {VIEW_TESTE}, {TAB_CLASSES} ou {V_VALID}: rode os notebooks 03m e 04m"
# fotografia dos arquivos que JÁ EXISTEM (tamanho e data) e proteções
def foto(raiz, rot): return {(rot + "/" + str(f.relative_to(raiz))): (f.stat().st_size, f.stat().st_mtime_ns) for f in Path(raiz).rglob("*") if f.is_file()}
SNAP_ANTES = {**foto(PASTA_RUN, "svm"), **foto(PASTA_RF_AJ, "rf_aj"), **foto(PASTA_RF_PAD, "rf_pad")}
instalar_travas(PASTA_RUN); instalar_travas(PASTA_RF_AJ); instalar_travas(PASTA_RF_PAD); silenciar_aviso_sklearn()
print(f"🔒 Proteções ativas: treino BLOQUEADO (SVM e RF); sobrescrita e remoção BLOQUEADAS em {PASTA_RUN.name}, {PASTA_RF_AJ.name} e {PASTA_RF_PAD.name} ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")

pc = json.loads((PASTA_RUN / "preprocessamento.json").read_text(encoding="utf-8")); cfg = json.loads((PASTA_RUN / "config_treino.json").read_text(encoding="utf-8")); cls_gravado = json.loads((PASTA_RUN / "classes.json").read_text(encoding="utf-8"))
CLASSES, COD = cls_gravado["classes"], cls_gravado["codigos"]; K = len(CLASSES); assert [COD[c] for c in CLASSES] == list(range(K)) and COD["BENIGN"] == 0
prep = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"]); CM = prep.colunas_modelo; assert CM == pc["colunas_modelo"]
assert colunas_view(VIEW_TESTE)[:2] == ["id_linha", "y"] and colunas_view(VIEW_TESTE)[2:] == CM, "as colunas da view de teste não coincidem com as do modelo (nome ou ordem)"
HP = {"ajustado": cfg["hiperparametros"], "padrao": cfg["padrao"]}
MODELOS = {n: {"arquivo": PASTA_RUN / a, "C": float(HP[n]["C"]), "gamma_fator": float(HP[n]["gamma_fator"]), "gamma": float(HP[n]["gamma"]), "pasta_teste": PASTA_RUN / f"teste_{n}"} for n, a in (("ajustado", "modelo_final_refit.joblib"), ("padrao", "modelo_padrao_refit.joblib"))}
ORDEM_MODELOS = [n for n in ("ajustado", "padrao") if MODELOS[n]["arquivo"].exists()]; assert "ajustado" in ORDEM_MODELOS
ROT = {"ajustado": "SVM ajustado", "padrao": "SVM padrão"}; SV = {}
for n in ORDEM_MODELOS:                                                                                  # confere cada modelo gravado e conta os vetores de suporte
    m = joblib.load(MODELOS[n]["arquivo"]); svm = m.named_steps["svm"]
    assert abs(svm.C - MODELOS[n]["C"]) < 1e-12 and abs(svm._gamma - MODELOS[n]["gamma"]) < 1e-12 and list(svm.classes_) == list(range(K)) and list(m.colunas_modelo_) == CM, f"o modelo {n} não corresponde à configuração gravada"
    SV[n] = int(svm.n_support_.sum()); del m
cl = consultar_num(f"SELECT codigo, classe, linhas_teste_sem_sobrep FROM {TAB_CLASSES} ORDER BY codigo"); assert cl.classe.tolist() == CLASSES
N_CLASSE = {c: int(n) for c, n in zip(cl.classe, cl.linhas_teste_sem_sobrep)}; N_TESTE = n_linhas(VIEW_TESTE); assert sum(N_CLASSE.values()) == N_TESTE
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
tt = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {TAB_TESTE} WHERE sobrepoe_treino = 0 GROUP BY label_multi"); tt["tipo"] = tt["tipo"].map(nome_tipo); tt = tt.groupby("tipo", as_index=False)["n"].sum()
FORA = {r.tipo: int(r.n) for r in tt.itertuples() if r.tipo not in COD}                                  # classes do teste que o modelo não conhece (ex.: Portmap)
# custo medido no 11m (vazão de 1 processo, na validação) → custo por par (linha × vetor de suporte)
custo11 = json.loads((PASTA_RUN / "custo_computacional.json").read_text(encoding="utf-8")); i1_ = next(r for r in custo11["inferencia_modelo_final"] if r["n_jobs"] == 1)
NS_PAR = 1e9 / (i1_["vazao_modelo_fluxos_s"] * custo11["modelo_final"]["n_vetores_suporte"])
ordem = ["roc_auc_ovr_macro", "acuracia_balanceada", "acuracia", "precisao_macro", "f1_macro", "f1_ponderado", "mcc"]; ESPECIAL = globals().get("CLASSE_ESPECIAL", "UDPLag"); I_ESP = COD.get(ESPECIAL)
print(f"SVM: {PASTA_RUN.name} | modelos: " + " | ".join(f"{ROT[n]} (C = {MODELOS[n]['C']:g}, gamma = {MODELOS[n]['gamma_fator']:g}×, {SV[n]:,} vetores de suporte)" for n in ORDEM_MODELOS))
print(f"RF ajustado (08m): {PASTA_RF_AJ.name} | RF padrão (06m): {PASTA_RF_PAD.name}")
print(f"Teste (sem sobreposição com o treino, só das {K} classes): {N_TESTE:,} linhas")
print("  " + " | ".join(f"{c}: {n:,} ({100 * n / N_TESTE:.3f}%)" for c, n in N_CLASSE.items())); print("Classes do teste que o modelo NÃO conhece (fora das métricas principais; análise à parte):", FORA or "nenhuma")
print(f"Custo de previsão medido no 11m: ~{NS_PAR:.0f} ns por par (linha × vetor de suporte)")
T_SEC = {}


🔒 Proteções ativas: treino BLOQUEADO (SVM e RF); sobrescrita e remoção BLOQUEADAS em svm_busca_20261107_013148, rfm_busca_20261102_001711 e rfm_20261101_210334 (672 arquivos já existem e serão conferidos no final); o servidor só é lido.
SVM: svm_busca_20261107_013148 | modelos: SVM ajustado (C = 100, gamma = 4×, 77,699 vetores de suporte) | SVM padrão (C = 1, gamma = 1×, 84,556 vetores de suporte)
RF ajustado (08m): rfm_busca_20261102_001711 | RF padrão (06m): rfm_20261101_210334
Teste (sem sobreposição com o treino, só das 7 classes): 7,216,487 linhas
  BENIGN: 56,216 (0.779%) | LDAP: 112,491 (1.559%) | MSSQL: 854,242 (11.837%) | NetBIOS: 91,175 (1.263%) | Syn: 4,887,060 (67.721%) | UDP: 1,214,746 (16.833%) | UDPLag: 557 (0.008%)
Classes do teste que o modelo NÃO conhece (fora das métricas principais; análise à parte): {'Portmap': 2702}
Custo de previsão medido no 11m: ~162 ns por par (linha × vetor de suporte)


## 2. Leitura do teste e vetores distintos
Lê o teste **uma vez** do servidor, em lotes, e identifica os **vetores distintos** (comparação exata, byte a byte, por uma chave de 128 bits). Guarda o **mapa linha → vetor distinto** e os vetores distintos em disco, mais uma **amostra de 3.000 linhas** com as suas features para a prova de equivalência. **Retomável.** Mostra quantos vetores distintos há e **estima o tempo da previsão**.

In [7]:
# ---------- 2.1 Leitura do teste e vetores distintos ----------
t_ini = time.perf_counter(); PASTA_MAPA, PASTA_UNIC = PASTA_RUN / "mapa_teste", PASTA_RUN / "vetores_unicos"; PASTA_MAPA.mkdir(exist_ok=True); PASTA_UNIC.mkdir(exist_ok=True)
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); D = len(CM); cols_sql = ", ".join("v." + q(c) for c in CM)
POS_AMOSTRA = np.sort(np.random.default_rng(SEMENTE).choice(N_TESTE, size=min(N_AMOSTRA_EQUIV, N_TESTE), replace=False))             # linhas que serão previstas diretamente (prova da equivalência)
def chaves(X):
    # Chave EXATA de cada linha: hash de 128 bits dos 8 × D bytes do vetor (float64). Vetores com chaves iguais são idênticos byte a byte (a chance de colisão é desprezível: ~2^-128).
    X = np.ascontiguousarray(X, dtype=np.float64)
    if X.shape[0] == 0: return []                                       # faixa de ids sem linhas
    mv = memoryview(X).cast("B"); w = X.shape[1] * 8
    return [hashlib.blake2b(mv[i * w:(i + 1) * w], digest_size=16).digest() for i in range(X.shape[0])]
tabela = {}; arqs_u = sorted(PASTA_UNIC.glob("u_*.npy"))
for f in arqs_u:                                                      # retomada: reconstrói a tabela de chaves a partir dos vetores já guardados
    for k in chaves(np.load(f)): tabela[k] = len(tabela)
offset = 0; feitos = sorted(PASTA_MAPA.glob("lote_*_de_*.npz"))
for f in feitos: offset += len(np.load(f)["y"])
barra = Barra(B, "Leitura do teste e vetores distintos"); barra.n = len(feitos); barra._mostrar(forcar=True); novos_lotes = 0
for j in range(len(feitos), B):
    arq_m = PASTA_MAPA / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq_m.exists(): continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B; onde_v = f"v.id_linha >= {lo}" + (f" AND v.id_linha < {hi}" if j < B - 1 else "")
    barra.avancar(0, f"lote {j + 1}/{B}: lendo... (memória {rss_mib():,.0f} MiB; vetores distintos até agora: {len(tabela):,})")
    df = ler_csv_com_espera(f"SELECT toString(v.id_linha) AS id, v.y, {cols_sql} FROM {VIEW_TESTE} AS v WHERE {onde_v}", ["id", "y"] + CM, texto=("id",))
    n = len(df); X = df[CM].to_numpy(float); y = df["y"].to_numpy(np.uint8); ids = df["id"].astype("uint64").to_numpy()
    if n: assert np.isfinite(X).all() and int(y.max()) < K, "valores não finitos ou classe fora do modelo no teste"
    uid = np.empty(n, np.int32); novos = []
    for i, k in enumerate(chaves(X)):
        u = tabela.get(k)
        if u is None: u = len(tabela); tabela[k] = u; novos.append(i)
        uid[i] = u
    if novos:
        arq_u = PASTA_UNIC / f"u_{j:03d}.npy"
        if not arq_u.exists():
            tmp = tmp_novo(arq_u)
            with open(tmp, "wb") as fh: np.save(fh, X[novos])
            os.replace(tmp, arq_u)
    sel = POS_AMOSTRA[(POS_AMOSTRA >= offset) & (POS_AMOSTRA < offset + n)] - offset
    tmp = tmp_novo(arq_m); np.savez_compressed(tmp, id=ids, y=y, uid=uid, n_novos=np.int64(len(novos)), amostra_pos=(sel + offset).astype(np.int64), amostra_X=X[sel]); os.replace(tmp, arq_m)
    offset += n; novos_lotes += 1; barra.avancar(1, f"lote {j + 1}/{B}: {n:,} linhas, {len(novos):,} vetores novos (distintos até agora: {len(tabela):,})"); del df, X
barra.fechar()
mapas = [np.load(f) for f in sorted(PASTA_MAPA.glob("lote_*_de_*.npz"))]; assert len(mapas) == B, f"o mapa tem {len(mapas)} lotes, esperava {B} (LOTE_LINHAS mudou entre execuções?)"
id_all, y_all, uid_all = (np.concatenate([m[c] for m in mapas]) for c in ("id", "y", "uid")); POS_A = np.concatenate([m["amostra_pos"] for m in mapas]); X_A = np.concatenate([m["amostra_X"] for m in mapas]); del mapas
ARQS_U = sorted(PASTA_UNIC.glob("u_*.npy")); N_POR_U = [len(np.load(f, mmap_mode="r")) for f in ARQS_U]; INICIO_U = np.concatenate([[0], np.cumsum(N_POR_U)]).astype(np.int64); N_UNICOS = int(INICIO_U[-1]); del tabela
assert len(y_all) == N_TESTE and N_UNICOS == int(uid_all.max()) + 1 and np.array_equal(np.sort(POS_A), POS_AMOSTRA), "o mapa do teste não fecha com a view"
assert all(int((y_all == COD[c]).sum()) == N_CLASSE[c] for c in CLASSES), "as linhas por classe do mapa não coincidem com a tabela do notebook 03m"
frac = N_UNICOS / N_TESTE
print(f"\n✔ {N_TESTE:,} linhas lidas | {N_UNICOS:,} vetores DISTINTOS ({100 * frac:.1f}% das linhas; {100 * (1 - frac):.1f}% são repetidas) | {len(ARQS_U)} arquivos de vetores | {fmt_tempo(time.perf_counter() - t_ini)}")
print("Estimativa do tempo de previsão (só os vetores distintos; 1 processo → com o paralelismo que a memória permitir):")
for n in ORDEM_MODELOS: print(f"  {ROT[n]}: {N_UNICOS:,} vetores × {SV[n]:,} vetores de suporte × {NS_PAR:.0f} ns = {fmt_tempo(N_UNICOS * SV[n] * NS_PAR * 1e-9)} com 1 processo (prever TODAS as {N_TESTE:,} linhas levaria {fmt_tempo(N_TESTE * SV[n] * NS_PAR * 1e-9)})")
T_SEC["2.1_mapa"] = time.perf_counter() - t_ini


  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: 1.10 GiB); esperando até 30s e tentando de novo (1/5)...
Leitura do teste e vetores distintos: concluído em 2min 10s

✔ 7,216,487 linhas lidas | 1,865,985 vetores DISTINTOS (25.9% das linhas; 74.1% são repetidas) | 145 arquivos de vetores | 2min 18s
Estimativa do tempo de previsão (só os vetores distintos; 1 processo → com o paralelismo que a memória permitir):
  SVM ajustado: 1,865,985 vetores × 77,699 vetores de suporte × 162 ns = 6h 31min com 1 processo (prever TODAS as 7,216,487 linhas levaria 25h 15min)
  SVM padrão: 1,865,985 vetores × 84,556 vetores de suporte × 162 ns = 7h 06min com 1 processo (prever TODAS as 7,216,487 linhas levaria 27h 29min)


## 3. Previsão dos vetores distintos (paralela, só se houver memória)
Prevê cada arquivo de vetores distintos **num processo à parte**, com **vários ao mesmo tempo só se houver memória livre** (a mesma regra do 11m). Cada arquivo previsto é salvo ao terminar: **se interromper, rode de novo**. Modelos já avaliados no teste são **pulados**.

In [8]:
# ---------- 3.1 Previsão dos vetores distintos (paralela, só se houver memória) ----------
PASTA_PREV = PASTA_RUN / "previsoes"; PASTA_PREV.mkdir(exist_ok=True)
def _prever_arquivo(caminho_modelo, caminho_X, caminho_saida):
    # Um processo NOVO por tarefa: carrega o modelo, prevê os vetores de UM arquivo (escores um-contra-resto, n x K) e grava o resultado antes de devolver os tempos.
    m = joblib.load(caminho_modelo); X = np.load(caminho_X)
    with Medidor() as med: sc = m.decision_function(X)
    tmp = Path(str(caminho_saida) + ".tmp.npy")
    with open(tmp, "wb") as fh: np.save(fh, sc.astype(np.float32))
    os.replace(tmp, caminho_saida)
    return {"n": int(len(X)), "tempo_s": med.relogio, "cpu_s": med.cpu, "pico_ram_mib": med.pico_mib}, None

def arq_prev(nome, j): return PASTA_PREV / nome / f"s_{j:03d}.npy"
def modelo_avaliado(nome): return (MODELOS[nome]["pasta_teste"] / "metricas_teste.json").exists() and not PERMITIR_REPETIR
STATS_PREV = {}
for nome in ORDEM_MODELOS:
    (PASTA_PREV / nome).mkdir(exist_ok=True)
    if modelo_avaliado(nome): print(f"↷ {ROT[nome]}: já avaliado no teste (metricas_teste.json existe): a previsão será pulada e o resultado recarregado"); continue
    pend = [j for j in range(len(ARQS_U)) if not arq_prev(nome, j).exists()]; prontos = len(ARQS_U) - len(pend)
    print(f"\n{ROT[nome]}: arquivos de vetores prontos: {prontos}/{len(ARQS_U)}" + (" (retomando)" if prontos else "") + f" | a prever agora: {len(pend)} ({int(sum(N_POR_U[j] for j in pend)):,} vetores)")
    if not pend: continue
    print(f"  Memória livre agora: {ram_livre_gib():.1f} GiB | cada processo precisa de ~{RAM_POR_PROCESSO_GIB:.1f} GiB × 1,3 mais {RAM_RESERVA_GIB:.1f} GiB de reserva → posso começar com {processos_seguros(PROCESSOS_PREVISAO_MAX, RAM_POR_PROCESSO_GIB)} processo(s) (teto {PROCESSOS_PREVISAO_MAX}; reavalio a memória a cada partida)")
    barra = Barra(len(ARQS_U), f"Previsão: {ROT[nome]}"); barra.n = prontos; barra._mostrar(forcar=True); soma_tarefas, t_sessao, feitas = 0.0, time.perf_counter(), []
    def lancar(j, nome=nome):
        ex = LokyPool(max_workers=1); return ex, ex.submit(_prever_arquivo, str(MODELOS[nome]["arquivo"]), str(ARQS_U[j]), str(arq_prev(nome, j)))
    def coletar(j, res, k_ini, nome=nome):
        global soma_tarefas
        assert res is not None, f"a tarefa {j} do {ROT[nome]} passou do limite de {fmt_tempo(LIMITE_TAREFA_S)}: aumente LIMITE_TAREFA_S"
        meta, _ = res; soma_tarefas += meta["tempo_s"]; feitas.append({**meta, "arquivo": j, "simultaneos_na_partida": k_ini}); barra.avancar(1, f"arquivo {j + 1}/{len(ARQS_U)}: {meta['n']:,} vetores em {fmt_tempo(meta['tempo_s'])} ({k_ini} simultâneo(s) na partida)")
    try: stats = rodar_tarefas(pend, lancar, coletar, barra, LIMITE_TAREFA_S, PROCESSOS_PREVISAO_MAX, RAM_POR_PROCESSO_GIB, f"previsão {ROT[nome]}", PARAR_APOS_TAREFAS)
    finally:
        rel = time.perf_counter() - t_sessao
        if feitas: pd.DataFrame([{"modelo": nome, "inicio": (datetime.now()).isoformat(timespec="seconds"), "relogio_s": rel, "tarefas": len(feitas), "vetores": int(sum(f["n"] for f in feitas)), "soma_tempos_tarefas_s": soma_tarefas}]).to_csv(PASTA_PREV / f"tempos_{nome}_{datetime.now():%Y%m%d_%H%M%S}.csv", index=False)       # um arquivo NOVO por sessão (nunca altera um existente)
    barra.fechar(); STATS_PREV[nome] = {**stats, "relogio_s": rel, "soma_tempos_tarefas_s": soma_tarefas, "ganho_paralelismo": soma_tarefas / rel if rel > 0 else float("nan")}
    print(f"✔ {ROT[nome]}: {len(feitas)} arquivos previstos em {fmt_tempo(rel)} | no máximo {stats['max_simultaneos']} processo(s) simultâneo(s) | ganho observado do paralelismo: {STATS_PREV[nome]['ganho_paralelismo']:.2f}× | memória por processo medida: ~{stats['ram_proc_final']:.2f} GiB" + (f" | tarefas interrompidas por memória: {stats['interrompidos_por_memoria']}" if stats["interrompidos_por_memoria"] else ""))
T_SEC["3.1_previsao"] = sum(s["relogio_s"] for s in STATS_PREV.values()) if STATS_PREV else 0.0



SVM ajustado: arquivos de vetores prontos: 0/145 | a prever agora: 145 (1,865,985 vetores)
  Memória livre agora: 1.0 GiB | cada processo precisa de ~0.6 GiB × 1,3 mais 1.0 GiB de reserva → posso começar com 1 processo(s) (teto 4; reavalio a memória a cada partida)


Previsão: SVM ajustado: concluído em 6h 17min
✔ SVM ajustado: 145 arquivos previstos em 6h 17min | no máximo 1 processo(s) simultâneo(s) | ganho observado do paralelismo: 0.98× | memória por processo medida: ~0.60 GiB

SVM padrão: arquivos de vetores prontos: 0/145 | a prever agora: 145 (1,865,985 vetores)
  Memória livre agora: 1.7 GiB | cada processo precisa de ~0.6 GiB × 1,3 mais 1.0 GiB de reserva → posso começar com 1 processo(s) (teto 4; reavalio a memória a cada partida)


Previsão: SVM padrão: concluído em 6h 37min
✔ SVM padrão: 145 arquivos previstos em 6h 37min | no máximo 1 processo(s) simultâneo(s) | ganho observado do paralelismo: 0.99× | memória por processo medida: ~0.60 GiB


## 4. Métricas no teste, por modelo
Espalha os escores dos vetores distintos para as **7.216.487 linhas**, **prova a equivalência** (3.000 linhas do teste previstas diretamente) e calcula as mesmas métricas do 06m e do 08m: gerais, por classe, matriz de confusão, `UDPLag` à parte, validação × teste e o `Portmap`. Cada arquivo só é criado se ainda não existir.

In [9]:
# ---------- 4.1 Métricas no teste, por modelo ----------
va = ler_view(V_VALID); va = va.iloc[np.argsort(va["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True); y_va, w_va = va["y"].to_numpy(int), va["peso"].to_numpy(float)
dfp = None
if FORA:                                                                # linhas das classes que o modelo não conhece (Portmap): lidas cruas e passadas pelo mesmo pré-processamento
    cols_raw = ", ".join(f"ifNull(toFloat64({q(c)}), nan) AS {q(c)}" for c in prep.features); XP = {}
    for classe, n_c in FORA.items():
        dfp_ = ler_csv_com_espera(f"SELECT {cols_raw} FROM {TAB_TESTE} WHERE label_multi IN ({lit(classe)}, {lit('DrDoS_' + classe)}) AND sobrepoe_treino = 0", prep.features); assert len(dfp_) == n_c, f"{classe}: leu {len(dfp_):,}, esperava {n_c:,}"
        XP[classe] = prep.transformar(dfp_[prep.features].to_numpy(float))
pcts = lambda a: [round(float(np.percentile(a, q_)), 3) for q_ in (10, 50, 90)]
R = {}                                                                  # resultados por modelo (para a comparação)

def avaliar(nome):
    t_sec0 = time.perf_counter(); pasta = MODELOS[nome]["pasta_teste"]; lock = pasta / "metricas_teste.json"; pasta.mkdir(exist_ok=True); r = {"nome": nome}
    if lock.exists() and not PERMITIR_REPETIR:                          # já avaliado: só recarrega (o teste não é gasto de novo)
        j = json.loads(lock.read_text(encoding="utf-8")); d = np.load(pasta / "predicoes_teste.npz")
        r.update(esc=j["metricas"], f1_sem=j["metricas_sem_classe_especial"]["f1_macro"], por_classe=j["por_classe"], cm=np.array([[j["matriz_confusao"][f"real {a}"][f"prev. {b}"] for b in CLASSES] for a in CLASSES], float),
                 id=d["id_linha"], y=d["y"], pred=d["pred"], portmap=j.get("classes_desconhecidas"), custo=j.get("custo"), recarregado=True); print(f"↷ {ROT[nome]}: resultado recarregado de {lock.relative_to(PASTA_RUN)}"); return r
    # escores dos vetores distintos (na ordem dos arquivos) → espalhados para as 7,2 milhões de linhas
    S_u = np.concatenate([np.load(arq_prev(nome, j)) for j in range(len(ARQS_U))]); assert S_u.shape == (N_UNICOS, K) and np.isfinite(S_u).all(), "escores incompletos ou não finitos"
    S_te = S_u[uid_all]; del S_u
    modelo = joblib.load(MODELOS[nome]["arquivo"]); t0 = time.perf_counter(); S_dir = modelo.decision_function(X_A); t_dir = time.perf_counter() - t0        # PROVA DA EQUIVALÊNCIA: 3.000 linhas do teste previstas diretamente
    pos_ord = np.argsort(POS_A, kind="stable"); dif_equiv = float(np.abs(S_te[POS_A[pos_ord]].astype(np.float64) - S_dir[pos_ord]).max()); acordo = float((S_te[POS_A[pos_ord]].argmax(1) == S_dir[pos_ord].argmax(1)).mean())
    print(f"\n{'✔' if acordo == 1.0 and dif_equiv < 1e-4 else '✘'} Equivalência ({ROT[nome]}): {len(POS_A):,} linhas do teste previstas DIRETAMENTE × resultado por vetores distintos: classes iguais em {100 * acordo:.2f}% das linhas, diferença máxima nos escores {dif_equiv:.1e} (float32 × float64)")
    r["equiv"] = {"n": int(len(POS_A)), "acordo": acordo, "dif_max": dif_equiv}; assert acordo == 1.0, "a previsão por vetores distintos NÃO reproduz a previsão direta"
    esc, arr = metricas_multi_grande(y_all, S_te); pred = S_te.argmax(axis=1).astype(np.uint8); cm = arr["cm"]
    f1_sem_esp = float(np.delete(arr["f1"], I_ESP).mean()) if I_ESP is not None else float("nan"); rec_sem_esp = float(np.nanmean(np.delete(arr["recall"], I_ESP))) if I_ESP is not None else float("nan")
    print(f"== {ROT[nome]}: métricas gerais no teste ({N_TESTE:,} linhas; mistura real das classes) =="); tab_te = pd.DataFrame({"teste": {k: esc[k] for k in ordem}}); display(tab_te.round(5))
    if I_ESP is not None: print(f"Métricas secundárias SEM a classe {ESPECIAL} na média (as previsões não mudam): F1 macro = {f1_sem_esp:.5f} | acurácia balanceada = {rec_sem_esp:.5f}")
    n = arr["suporte"].astype(int); acertos = np.diag(cm).astype(int); ic = [wilson(int(acertos[i]), int(n[i])) for i in range(K)]
    tab_cls_te = pd.DataFrame({"linhas no teste": n, "% do teste": 100 * n / n.sum(), "recall": arr["recall"], "IC95% recall (inf.)": [a for a, _ in ic], "IC95% recall (sup.)": [b for _, b in ic], "precisão": arr["precisao"], "F1": arr["f1"],
                               "ROC-AUC um-contra-resto": arr["auc"], "previstas como a classe": arr["previstas"].astype(int)}, index=CLASSES)
    print(f"\n== {ROT[nome]}: por classe (o IC95% é otimista: o teste tem ~89% de linhas repetidas) =="); display(tab_cls_te.round(5))
    cm_te = pd.DataFrame(cm.astype(int), index=[f"real {c}" for c in CLASSES], columns=[f"prev. {c}" for c in CLASSES]); cm_lin = pd.DataFrame(cm / cm.sum(axis=1, keepdims=True), index=cm_te.index, columns=cm_te.columns)
    cm_col = pd.DataFrame(np.divide(cm, cm.sum(axis=0, keepdims=True), out=np.zeros_like(cm), where=cm.sum(axis=0, keepdims=True) > 0), index=cm_te.index, columns=cm_te.columns)
    print(f"\n== {ROT[nome]}: matriz de confusão no teste (linhas = classe real; colunas = classe prevista) =="); display(cm_te); print("== Em fração da classe REAL (a diagonal é o recall) =="); display(cm_lin.round(4))
    print("== Em fração da classe PREVISTA (a diagonal é a precisão) =="); display(cm_col.round(4))
    fora = sorted([(CLASSES[i], CLASSES[j], int(cm[i, j]), cm[i, j] / cm[i].sum()) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0], key=lambda x: -x[2])[:10]
    print("Confusões mais frequentes (real → previsto): " + " | ".join(f"{a}→{b}: {nn:,} ({100 * f:.2f}% de {a})" for a, b, nn, f in fora))
    if I_ESP is not None:
        comp = pd.DataFrame({"linhas": cm[:, I_ESP].astype(int), f"% das previsões de {ESPECIAL}": 100 * cm[:, I_ESP] / max(cm[:, I_ESP].sum(), 1)}, index=CLASSES); print(f"\n== De onde vêm as {int(cm[:, I_ESP].sum()):,} previsões de {ESPECIAL} (o teste só tem {int(n[I_ESP]):,} linhas reais dessa classe) =="); display(comp.round(3))
    # validação × teste (a validação é do mesmo dia do treino; escores do treino completo no ajuste, gravados pelo 11m)
    sc_val = np.load(PASTA_RUN / "fits" / f"{nome_cfg(MODELOS[nome]['C'], MODELOS[nome]['gamma_fator'])}_completo.npz")["scores"]; assert sc_val.shape == (len(y_va), K)
    ev, av = metricas_multi(y_va, sc_val, None); ev = {("roc_auc_ovr_macro" if k == "roc_auc_ovr_macro" else k): v for k, v in ev.items()}
    rec_v, pre_v = av["recall"], av["precisao"]; f1_v = av["f1"]
    vs_cls = pd.DataFrame({"recall (validação)": rec_v, "recall (teste)": arr["recall"], "Δ recall": arr["recall"] - rec_v, "precisão (validação)": pre_v, "precisão (teste)": arr["precisao"], "F1 (validação)": f1_v, "F1 (teste)": arr["f1"]}, index=CLASSES)
    vs_ger = pd.DataFrame({"validação (sem pesos)": {k: ev[k] for k in ordem}, "teste": {k: esc[k] for k in ordem}}); vs_ger["Δ"] = vs_ger["teste"] - vs_ger["validação (sem pesos)"]
    print(f"\n== {ROT[nome]}: validação × teste: geral =="); display(vs_ger.round(5)); print("== Validação × teste: por classe =="); display(vs_cls.round(4))
    # classes desconhecidas (Portmap)
    PORT = pd.DataFrame(); port_json = {}
    for classe, Xp in XP.items():
        Sp = modelo.decision_function(Xp).astype(np.float32); pp = Sp.argmax(axis=1); smax = Sp.max(axis=1); votos = np.rint(smax).astype(int)
        linhas_p = [{"classe desconhecida": classe, "prevista como": CLASSES[k], "linhas": int((pp == k).sum()), "%": 100 * float((pp == k).mean()), "escore médio da classe": float(Sp[:, k].mean()), "escore máximo mediano (quando prevista)": float(np.median(smax[pp == k])) if (pp == k).any() else np.nan} for k in range(K)]
        print(f"\n== {ROT[nome]} — {classe}: {len(Xp):,} linhas (o modelo nunca viu esta classe): para qual classe é empurrada =="); display(pd.DataFrame(linhas_p).drop(columns="classe desconhecida").set_index("prevista como").round(4))
        pa = S_te[np.arange(len(pred)), pred]; ok_ = (pred == y_all)
        print(f"Escore da classe escolhida (percentis 10/50/90):  {classe}: {pcts(smax)}  | acertos das classes conhecidas: {pcts(pa[ok_])}  | erros das classes conhecidas: {pcts(pa[~ok_]) if (~ok_).any() else 'sem erros'}")
        print(f"(o escore da classe escolhida ≈ número de duelos de pares vencidos, de 0 a 6; mediana {classe}: {int(np.median(votos))})")
        PORT = pd.concat([PORT, pd.DataFrame(linhas_p)], ignore_index=True); port_json[classe] = {c: int((pp == COD[c]).sum()) for c in CLASSES}
    t_prev = STATS_PREV.get(nome, {}); custo = {"vetores_distintos": N_UNICOS, "linhas": N_TESTE, "vetores_de_suporte": SV[nome], "relogio_previsao_s": t_prev.get("relogio_s"), "soma_tempos_tarefas_s": t_prev.get("soma_tempos_tarefas_s"), "ganho_paralelismo": t_prev.get("ganho_paralelismo"),
                                                  "max_simultaneos": t_prev.get("max_simultaneos"), "vazao_vetores_s": (N_UNICOS / t_prev["relogio_s"]) if t_prev.get("relogio_s") else None, "vazao_linhas_equivalente_s": (N_TESTE / t_prev["relogio_s"]) if t_prev.get("relogio_s") else None, "equivalencia": r["equiv"]}
    p_pred = S_te[np.arange(len(pred)), pred]; p_real = S_te[np.arange(len(y_all)), y_all]
    gravar(pasta / "predicoes_teste.npz", lambda p_: np.savez_compressed(p_, id_linha=id_all, y=y_all, pred=pred, p_pred=p_pred, p_real=p_real, classes=np.array(CLASSES)))
    gravar_csv(tab_te.reset_index(names="metrica"), pasta / "metricas_teste.csv", index=False); gravar_csv(tab_cls_te.reset_index(names="classe"), pasta / "metricas_por_classe_teste.csv", index=False)
    gravar_csv(cm_te.reset_index(names="classe_real"), pasta / "matriz_confusao_teste.csv", index=False); gravar_csv(cm_lin.reset_index(names="classe_real"), pasta / "matriz_confusao_teste_fracao_da_real.csv", index=False)
    gravar_csv(vs_cls.reset_index(names="classe"), pasta / "validacao_vs_teste.csv", index=False); gravar_csv(vs_ger.reset_index(names="metrica"), pasta / "validacao_vs_teste_geral.csv", index=False)
    if len(PORT): gravar_csv(PORT, pasta / "portmap_teste.csv", index=False)
    gravar_json(custo, pasta / "custo_inferencia_teste.json")
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(13, 5.2)); im = ax[0].imshow(cm_lin.values, cmap="Blues", vmin=0, vmax=1); ax[0].set_xticks(range(K)); ax[0].set_yticks(range(K)); ax[0].set_xticklabels(CLASSES, rotation=45, ha="right"); ax[0].set_yticklabels(CLASSES)
        for i in range(K):
            for j in range(K): ax[0].text(j, i, f"{cm_lin.values[i, j]:.2f}" if cm_lin.values[i, j] >= 0.005 else "", ha="center", va="center", fontsize=8, color="white" if cm_lin.values[i, j] > 0.5 else "black")
        ax[0].set_xlabel("classe prevista"); ax[0].set_ylabel("classe real"); ax[0].set_title(f"{ROT[nome]}: matriz de confusão no teste (fração da classe real)"); fig.colorbar(im, ax=ax[0], fraction=0.046)
        x = np.arange(K); ax[1].bar(x - 0.2, vs_cls["recall (validação)"], 0.4, label="validação (mesmo dia)"); ax[1].bar(x + 0.2, vs_cls["recall (teste)"], 0.4, label="teste (outro dia)"); ax[1].set_xticks(x); ax[1].set_xticklabels(CLASSES, rotation=45, ha="right")
        ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("recall"); ax[1].set_title("Recall por classe: validação × teste"); ax[1].legend(loc="lower right"); ax[1].grid(axis="y", alpha=.3); fig.tight_layout(); gravar(pasta / "teste_multiclasse.png", lambda p_: fig.savefig(p_, dpi=130)); plt.close(fig)
    except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")
    json_lock = {"run_id": cfg["run_id"], "modelo": nome, "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": N_TESTE, "classes": CLASSES, "metricas": esc, "metricas_sem_classe_especial": {"classe": ESPECIAL, "f1_macro": f1_sem_esp, "acuracia_balanceada": rec_sem_esp},
                 "por_classe": tab_cls_te.to_dict(orient="index"), "matriz_confusao": cm_te.to_dict(orient="index"), "validacao_vs_teste": vs_ger.to_dict(orient="index"), "classes_desconhecidas": port_json, "custo": custo}
    r.update(esc=esc, f1_sem=f1_sem_esp, por_classe=tab_cls_te.to_dict(orient="index"), cm=cm, id=id_all, y=y_all, pred=pred, portmap=port_json, custo=custo, recarregado=False, lock=(pasta, json_lock), vs_ger=vs_ger)
    T_SEC[f"4.1_{nome}"] = time.perf_counter() - t_sec0; del S_te, p_pred, p_real; return r

for nome in ORDEM_MODELOS: R[nome] = avaliar(nome)


  conexão instável (RemoteDisconnected: Remote end closed connection without response); nova tentativa em 5s...

✔ Equivalência (SVM ajustado): 3,000 linhas do teste previstas DIRETAMENTE × resultado por vetores distintos: classes iguais em 100.00% das linhas, diferença máxima nos escores 2.4e-07 (float32 × float64)
== SVM ajustado: métricas gerais no teste (7,216,487 linhas; mistura real das classes) ==


,teste
roc_auc_ovr_macro,0.89974
acuracia_balanceada,0.73350
acuracia,0.56457
precisao_macro,0.68179
f1_macro,0.60222
f1_ponderado,0.68953
mcc,0.51724


Métricas secundárias SEM a classe UDPLag na média (as previsões não mudam): F1 macro = 0.70256 | acurácia balanceada = 0.79800

== SVM ajustado: por classe (o IC95% é otimista: o teste tem ~89% de linhas repetidas) ==


,linhas no teste,% do teste,recall,IC95% recall (inf.),IC95% recall (sup.),precisão,F1,ROC-AUC um-contra-resto,previstas como a classe
BENIGN,56216,0.77899,0.98502,0.98398,0.98599,0.86038,0.91849,0.99605,64360
LDAP,112491,1.55881,0.98235,0.98156,0.98310,0.78296,0.87139,0.99356,141137
MSSQL,854242,11.83737,0.37480,0.37378,0.37583,0.98415,0.54286,0.99778,325328
NetBIOS,91175,1.26343,0.99873,0.99847,0.99894,0.14506,0.25333,0.99104,627723
Syn,4887060,67.72076,0.47355,0.47311,0.47399,0.99998,0.64273,0.99411,2314323
UDP,1214746,16.83293,0.97357,0.97328,0.97385,0.99991,0.98656,0.99861,1182741
UDPLag,557,0.00772,0.34650,0.30815,0.38695,0.00008,0.00015,0.32702,2560875



== SVM ajustado: matriz de confusão no teste (linhas = classe real; colunas = classe prevista) ==


,prev. BENIGN,prev. LDAP,prev. MSSQL,prev. NetBIOS,prev. Syn,prev. UDP,prev. UDPLag
real BENIGN,55374,36,66,692,3,8,37
real LDAP,54,110505,614,1249,38,0,31
real MSSQL,198,30505,320173,503301,2,33,30
real NetBIOS,53,0,18,91059,1,4,40
real Syn,8553,88,1473,2371,2314276,8,2560291
real UDP,85,1,2819,28950,1,1182637,253
real UDPLag,43,2,165,101,2,51,193


... (27 outputs left out)

,recall (validação),recall (teste),Δ recall,precisão (validação),precisão (teste),F1 (validação),F1 (teste)
BENIGN,0.9959,0.9884,-0.0075,0.9855,0.9333,0.9906,0.9601
LDAP,0.9507,0.9847,0.0341,0.8920,0.7476,0.9204,0.8500
MSSQL,0.8984,0.3835,-0.5149,0.8165,0.9616,0.8555,0.5483
NetBIOS,0.5380,0.9964,0.4584,0.7857,0.1463,0.6386,0.2552
Syn,0.9729,0.5273,-0.4456,0.9347,0.9999,0.9534,0.6904
UDP,0.9816,0.9596,-0.0220,0.8166,0.9998,0.8915,0.9793
UDPLag,0.7033,0.0000,-0.7033,0.9642,0.0000,0.8133,0.0000



== SVM padrão — Portmap: 2,702 linhas (o modelo nunca viu esta classe): para qual classe é empurrada ==


,linhas,%,escore médio da classe,escore máximo mediano (quando prevista)
prevista como,,,,
BENIGN,125,4.6262,1.5605,6.2858
LDAP,0,0.0000,2.0504,NaN
MSSQL,90,3.3309,4.2778,6.3000
NetBIOS,2316,85.7143,6.0452,6.2980
Syn,167,6.1806,0.4704,6.2863
UDP,4,0.1480,3.3535,6.3044
UDPLag,0,0.0000,3.2505,NaN


Escore da classe escolhida (percentis 10/50/90):  Portmap: [6.287, 6.298, 6.299]  | acertos das classes conhecidas: [6.281, 6.286, 6.303]  | erros das classes conhecidas: [6.284, 6.284, 6.296]
(o escore da classe escolhida ≈ número de duelos de pares vencidos, de 0 a 6; mediana Portmap: 6)


## 5. Comparação: SVM × RF
Os quatro modelos (RF padrão, RF ajustado, SVM padrão, SVM ajustado) **nas mesmas linhas**: métricas gerais, a faixa das 5 sementes do RF (09m), recall, precisão, F1 e ROC-AUC por classe, o desempenho **ataque × BENIGN**, a comparação **linha a linha** com o RF do mesmo tipo e o destino das linhas `Syn`. É **descritiva**: o teste tem uma só partição e muitas linhas repetidas.

In [10]:
# ---------- 5.1 Comparação: SVM × RF (padrão e ajustado) ----------
rj = lambda p: json.loads((p / "metricas_teste.json").read_text(encoding="utf-8"))
ref_pad, ref_aj = rj(PASTA_RF_PAD), rj(PASTA_RF_AJ); assert ref_pad["n_teste"] == ref_aj["n_teste"] == N_TESTE and ref_pad["classes"] == ref_aj["classes"] == CLASSES, "o 06m e o 08m usaram outro teste ou outras classes"
cm_de = lambda j: np.array([[j["matriz_confusao"][f"real {a}"][f"prev. {b}"] for b in CLASSES] for a in CLASSES], float)
M4 = {"RF padrão (06m)": {"esc": ref_pad["metricas"], "f1_sem": ref_pad["metricas_sem_classe_especial"]["f1_macro"], "por_classe": ref_pad["por_classe"], "cm": cm_de(ref_pad)},
      "RF ajustado (08m)": {"esc": ref_aj["metricas"], "f1_sem": ref_aj["metricas_sem_classe_especial"]["f1_macro"], "por_classe": ref_aj["por_classe"], "cm": cm_de(ref_aj)}}
for n_ in ORDEM_MODELOS[::-1]: M4[ROT[n_]] = {k: R[n_][k] for k in ("esc", "f1_sem", "por_classe", "cm")}
print(f"Modelos comparados (mesmas {N_TESTE:,} linhas): {list(M4)}")
ger = pd.DataFrame({nome: {**{k: m["esc"][k] for k in ordem}, "f1_macro_sem_" + ESPECIAL: m["f1_sem"]} for nome, m in M4.items()}); print("\n== Métricas gerais no teste =="); display(ger.round(5))
seeds_csv = PASTA_RF_AJ / "estudo_sementes" / "resultados_por_semente.csv"; SEEDS = None
if seeds_csv.exists():                                                                       # a faixa das 5 sementes do RF (notebook 09m)
    SEEDS = pd.read_csv(seeds_csv); mt = ["roc_auc", "acuracia_balanceada", "f1_macro", "f1_macro_sem_UDPLag", "mcc", "recall_Syn", "recall_UDPLag", "ataque_recall", "syn_para_BENIGN_%"]
    fx = pd.DataFrame({f"RF {c} (média; mín–máx de {int((SEEDS.config == ('padrao' if c == 'padrão' else 'ajustado')).sum())} sementes)": {m: f"{SEEDS[SEEDS.config == ('padrao' if c == 'padrão' else 'ajustado')][m].mean():.4f} ({SEEDS[SEEDS.config == ('padrao' if c == 'padrão' else 'ajustado')][m].min():.4f}–{SEEDS[SEEDS.config == ('padrao' if c == 'padrão' else 'ajustado')][m].max():.4f})" for m in mt} for c in ("padrão", "ajustado")})
    print("\n== O RF com várias sementes (notebook 09m): o SVM é determinístico e tem uma nota só =="); display(fx)
else: print("\n(a tabela do estudo de sementes do RF, notebook 09m, não foi encontrada: a faixa das sementes foi omitida)")
cmp_cls = {}
for m in ("recall", "precisão", "F1", "ROC-AUC um-contra-resto"): cmp_cls[m] = pd.DataFrame({nome: [mm["por_classe"][c][m] for c in CLASSES] for nome, mm in M4.items()}, index=CLASSES)
for m, d in cmp_cls.items(): print(f"\n== {m} por classe =="); display(d.round(4))
def colapsa(cm_):
    cm_ = np.asarray(cm_, float); tn, fp, fn, tp = cm_[0, 0], cm_[0, 1:].sum(), cm_[1:, 0].sum(), cm_[1:, 1:].sum(); den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {"recall dos ataques": tp / (tp + fn), "falso alarme (BENIGN → ataque)": fp / (fp + tn), "MCC ataque × BENIGN": (tp * tn - fp * fn) / den if den > 0 else 0.0, "ataques previstos como BENIGN": int(fn), "Syn previsto como BENIGN (%)": 100 * cm_[COD["Syn"], 0] / cm_[COD["Syn"]].sum(), "Syn previsto como UDPLag (%)": 100 * cm_[COD["Syn"], COD["UDPLag"]] / cm_[COD["Syn"]].sum()}
colap = pd.DataFrame({nome: colapsa(mm["cm"]) for nome, mm in M4.items()}); print("\n== Ataque × BENIGN (as 6 classes de ataque juntas; cálculo posterior, descritivo) =="); display(colap.drop(index="ataques previstos como BENIGN").round(5))
print("Ataques previstos como BENIGN: " + " | ".join(f"{n_}: {int(v):,}" for n_, v in colap.loc["ataques previstos como BENIGN"].items()))
# linha a linha: cada SVM contra o RF do mesmo tipo (padrão com padrão, ajustado com ajustado), alinhando pelo id da linha
def alinhar(pasta_rf, r):
    d0 = np.load(pasta_rf / "predicoes_teste.npz"); o0, o1 = np.argsort(d0["id_linha"], kind="stable"), np.argsort(r["id"], kind="stable"); assert np.array_equal(d0["id_linha"][o0], r["id"][o1]), "o RF e o SVM não têm as mesmas linhas"
    assert np.array_equal(d0["y"][o0], r["y"][o1]); return d0["y"][o0], d0["pred"][o0], r["pred"][o1]
PAR, SYNT = {}, {}
for n_, pasta_rf, rotulo_rf in (("ajustado", PASTA_RF_AJ, "RF ajustado"), ("padrao", PASTA_RF_PAD, "RF padrão")):
    if n_ not in R: continue
    y0, p_rf, p_svm = alinhar(pasta_rf, R[n_]); ok_rf, ok_sv = p_rf == y0, p_svm == y0; linhas = []
    for nome, msk in [("todas as classes", np.ones(len(y0), bool))] + [(c, y0 == COD[c]) for c in CLASSES]:
        a, b, c_, d_ = int((ok_rf & ok_sv & msk).sum()), int((~ok_rf & ok_sv & msk).sum()), int((ok_rf & ~ok_sv & msk).sum()), int((~ok_rf & ~ok_sv & msk).sum())
        linhas.append({"classe real": nome, "linhas": int(msk.sum()), "certas nos dois": a, f"só o {ROT[n_]} acerta": b, f"só o {rotulo_rf} acerta": c_, "erram nos dois": d_, f"saldo ({ROT[n_]} − {rotulo_rf})": b - c_})
    PAR[n_] = pd.DataFrame(linhas); print(f"\n== Linha a linha: {ROT[n_]} × {rotulo_rf} (mesmas linhas nos dois) =="); display(PAR[n_])
    ms = (y0 == COD["Syn"]); SYNT[n_] = pd.crosstab(pd.Series(np.array(CLASSES)[p_rf[ms]], name=f"{rotulo_rf} previu"), pd.Series(np.array(CLASSES)[p_svm[ms]], name=f"{ROT[n_]} previu")); print(f"== Linhas reais Syn: para onde mudou a previsão ({rotulo_rf} → {ROT[n_]}) =="); display(SYNT[n_]); del y0, p_rf, p_svm, ok_rf, ok_sv
g_ = cmp_cls["recall"]; print(f"\nRecall: maior vantagem do {ROT['ajustado']} sobre o RF ajustado: {(g_[ROT['ajustado']] - g_['RF ajustado (08m)']).max():+.4f} ({(g_[ROT['ajustado']] - g_['RF ajustado (08m)']).idxmax()}) | maior desvantagem: {(g_[ROT['ajustado']] - g_['RF ajustado (08m)']).min():+.4f} ({(g_[ROT['ajustado']] - g_['RF ajustado (08m)']).idxmin()})")


Modelos comparados (mesmas 7,216,487 linhas): ['RF padrão (06m)', 'RF ajustado (08m)', 'SVM padrão', 'SVM ajustado']

== Métricas gerais no teste ==


,RF padrão (06m),RF ajustado (08m),SVM padrão,SVM ajustado
roc_auc_ovr_macro,0.91328,0.92483,0.88547,0.89974
acuracia_balanceada,0.69079,0.67714,0.69141,0.73350
acuracia,0.39168,0.33847,0.59963,0.56457
precisao_macro,0.69269,0.56041,0.68408,0.68179
f1_macro,0.56262,0.41477,0.61190,0.60222
f1_ponderado,0.51081,0.42581,0.72127,0.68953
mcc,0.43242,0.36879,0.53467,0.51724
f1_macro_sem_UDPLag,0.65637,0.48386,0.71388,0.70256



== O RF com várias sementes (notebook 09m): o SVM é determinístico e tem uma nota só ==


,RF padrão (média; mín–máx de 5 sementes),RF ajustado (média; mín–máx de 5 sementes)
roc_auc,0.9136 (0.9072–0.9196),0.9247 (0.9224–0.9260)
acuracia_balanceada,0.6843 (0.6775–0.6908),0.6731 (0.6697–0.6771)
f1_macro,0.4834 (0.4183–0.5626),0.4116 (0.4085–0.4148)
f1_macro_sem_UDPLag,0.5639 (0.4880–0.6564),0.4802 (0.4766–0.4839)
mcc,0.4022 (0.3666–0.4324),0.3651 (0.3616–0.3688)
recall_Syn,0.2029 (0.1668–0.2445),0.1505 (0.1366–0.1617)
recall_UDPLag,0.4029 (0.3914–0.4219),0.3418 (0.3196–0.3573)
ataque_recall,0.8833 (0.7210–0.9994),0.6858 (0.6753–0.7255)
syn_para_BENIGN_%,17.0855 (0.0745–40.8618),46.0203 (40.2092–47.5667)



== recall por classe ==


,RF padrão (06m),RF ajustado (08m),SVM padrão,SVM ajustado
BENIGN,0.9824,0.9806,0.9884,0.9850
LDAP,0.9632,0.9926,0.9847,0.9823
MSSQL,0.3681,0.3411,0.3835,0.3748
NetBIOS,0.9990,0.9987,0.9964,0.9987
Syn,0.2445,0.1617,0.5273,0.4736
UDP,0.8744,0.9081,0.9596,0.9736
UDPLag,0.4039,0.3573,0.0000,0.3465


... (13 outputs left out)

,classe real,linhas,certas nos dois,só o SVM padrão acerta,só o RF padrão acerta,erram nos dois,saldo (SVM padrão − RF padrão)
0,todas as classes,7216487,2472104,1855094,354420,2534869,1500674
1,BENIGN,56216,54594,971,633,18,338
2,LDAP,112491,107169,3603,1182,537,2421
3,MSSQL,854242,269334,58274,45112,481522,13162
4,NetBIOS,91175,90828,16,257,74,-241
5,Syn,4887060,908552,1668171,286515,2023822,1381656
6,UDP,1214746,1041627,124059,20496,28564,103563
7,UDPLag,557,0,0,225,332,-225


== Linhas reais Syn: para onde mudou a previsão (RF padrão → SVM padrão) ==


SVM padrão previu,BENIGN,LDAP,MSSQL,NetBIOS,Syn,UDP,UDPLag
RF padrão previu,,,,,,,
BENIGN,800,1,0,88,2547,0,206
LDAP,57,0,0,24,58,0,0
MSSQL,110,0,775,3,21,0,0
NetBIOS,1027,0,673,435,221,0,4
Syn,740,0,0,20,908552,0,285755
UDP,7,0,37,2,6,8,0
UDPLag,764,0,143,392,1665318,0,2018266



Recall: maior vantagem do SVM ajustado sobre o RF ajustado: +0.3119 (Syn) | maior desvantagem: -0.0108 (UDPLag)


## 6. Gravação e conferências
Grava a comparação (**só arquivos novos**) e confere: que as matrizes fecham, que a **previsão por vetores distintos reproduz 100% a previsão direta**, que nenhum modelo foi treinado e que **nenhum arquivo pré-existente** das pastas do SVM e do RF foi alterado. **O `metricas_teste.json` de cada modelo é gravado por último**, só se todas as conferências passarem: é ele que aciona a trava de uso único.

In [11]:
# ---------- 6.1 Gravação e conferências ----------
gravar_csv(ger.reset_index(names="metrica"), PASTA_RUN / "comparacao_svm_vs_rf_geral.csv", index=False); gravar_csv(colap.reset_index(names="metrica"), PASTA_RUN / "comparacao_svm_vs_rf_ataque_benign.csv", index=False)
for m, d in cmp_cls.items(): gravar_csv(d.reset_index(names="classe"), PASTA_RUN / f"comparacao_svm_vs_rf_{ {'recall': 'recall', 'precisão': 'precisao', 'F1': 'f1', 'ROC-AUC um-contra-resto': 'roc_auc'}[m] }_por_classe.csv", index=False)
for n_, d in PAR.items(): gravar_csv(d, PASTA_RUN / f"comparacao_svm_vs_rf_linha_a_linha_{n_}.csv", index=False); gravar_csv(SYNT[n_].reset_index(), PASTA_RUN / f"comparacao_svm_vs_rf_syn_transicoes_{n_}.csv", index=False)
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
cm_total = lambda cm_: int(cm_.sum()) == N_TESTE and all(int(cm_[COD[c]].sum()) == N_CLASSE[c] for c in CLASSES)
verifica("Todos os modelos do SVM têm resultado no teste", all(n_ in R for n_ in ORDEM_MODELOS), f"({', '.join(ROT[n_] for n_ in ORDEM_MODELOS)})")
verifica("As matrizes de confusão fecham com o total e com as linhas por classe, em todos os modelos (SVM e RF)", all(cm_total(mm["cm"]) for mm in M4.values()))
verifica("Equivalência: a previsão por vetores distintos reproduz 100% a previsão direta de linhas do teste (prova por amostra)", all(R[n_]["equiv"]["acordo"] == 1.0 for n_ in ORDEM_MODELOS if "equiv" in R[n_]), f"({len(POS_A):,} linhas; " + "; ".join(f"{ROT[n_]}: dif. máx. {R[n_]['equiv']['dif_max']:.1e}" for n_ in ORDEM_MODELOS if "equiv" in R[n_]) + ")")
verifica("O mapa do teste fecha: cada linha aponta para um vetor distinto válido, e os vetores distintos somam o total dos arquivos", int(uid_all.min()) == 0 and int(uid_all.max()) + 1 == N_UNICOS == int(sum(N_POR_U)) and N_UNICOS <= N_TESTE, f"({N_UNICOS:,} vetores distintos em {N_TESTE:,} linhas)")
verifica("Todas as previsões dos vetores distintos existem, para cada modelo avaliado agora", all(all(arq_prev(n_, j).exists() for j in range(len(ARQS_U))) for n_ in ORDEM_MODELOS if not R[n_]["recarregado"]))
verifica("Classes desconhecidas: todas as linhas foram lidas e previstas", all(len(XP[c]) == FORA[c] for c in FORA) if FORA else True, f"({FORA})" if FORA else "(nenhuma)")
verifica("Nenhum modelo foi treinado neste notebook (o treino está bloqueado e nenhuma tentativa foi feita)", len(TREINOS_BLOQUEADOS) == 0 and getattr(SVC.fit, "_bloqueado", False))
_depois = {**foto(PASTA_RUN, "svm"), **foto(PASTA_RF_AJ, "rf_aj"), **foto(PASTA_RF_PAD, "rf_pad")}; _mexidos = [n for n, v in SNAP_ANTES.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES)} arquivos que já existiam (pastas do SVM, do RF 07m/08m e do RF 05m/06m) foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("Arquivos esperados gravados, para cada modelo", all((MODELOS[n_]["pasta_teste"] / f).exists() for n_ in ORDEM_MODELOS if not R[n_]["recarregado"] for f in ("predicoes_teste.npz", "metricas_teste.csv", "metricas_por_classe_teste.csv", "matriz_confusao_teste.csv", "validacao_vs_teste.csv", "custo_inferencia_teste.json")) and (PASTA_RUN / "comparacao_svm_vs_rf_geral.csv").exists())
assert all(ok), "há checagens reprovadas acima (nenhum metricas_teste.json foi gravado, então o teste pode ser refeito: os escores dos vetores distintos já gravados são reaproveitados)"
for n_ in ORDEM_MODELOS:                                                                                                  # o metricas_teste.json de cada modelo é gravado POR ÚLTIMO: é ele que aciona a trava de uso único
    if not R[n_]["recarregado"]: pasta_, json_lock = R[n_]["lock"]; gravar_json(json_lock, pasta_ / "metricas_teste.json")
reg = {"run_id": cfg["run_id"], "data_teste": datetime.now().isoformat(timespec="seconds"), "n_teste": N_TESTE, "vetores_distintos": N_UNICOS, **{f"{n_}_{k}": R[n_]["esc"][k] for n_ in ORDEM_MODELOS for k in ("roc_auc_ovr_macro", "f1_macro", "mcc")}, **{f"{n_}_recall_{c}": R[n_]["por_classe"][c]["recall"] for n_ in ORDEM_MODELOS for c in CLASSES}, "pasta": str(PASTA_RUN)}
caminho_reg = PASTA_BASE / "registro_testes.csv"; pd.DataFrame([reg]).to_csv(caminho_reg, mode="a", header=not caminho_reg.exists(), index=False)
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_RUN}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print(f"\nArquivos NOVOS criados nesta execução ({len(CRIADOS)}): {', '.join(CRIADOS[:14])}{' ...' if len(CRIADOS) > 14 else ''}")
if MANTIDOS: print(f"Arquivos que já existiam e foram MANTIDOS ({len(MANTIDOS)}): {', '.join(MANTIDOS[:14])}{' ...' if len(MANTIDOS) > 14 else ''}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, os arquivos comparacao_svm_vs_rf_*.csv, metricas_por_classe_teste.csv e matriz_confusao_teste.csv de cada modelo, e teste_multiclasse.png).")


✔ Todos os modelos do SVM têm resultado no teste (SVM ajustado, SVM padrão)
✔ As matrizes de confusão fecham com o total e com as linhas por classe, em todos os modelos (SVM e RF) 
✔ Equivalência: a previsão por vetores distintos reproduz 100% a previsão direta de linhas do teste (prova por amostra) (3,000 linhas; SVM ajustado: dif. máx. 2.4e-07; SVM padrão: dif. máx. 2.4e-07)
✔ O mapa do teste fecha: cada linha aponta para um vetor distinto válido, e os vetores distintos somam o total dos arquivos (1,865,985 vetores distintos em 7,216,487 linhas)
✔ Todas as previsões dos vetores distintos existem, para cada modelo avaliado agora 
✔ Classes desconhecidas: todas as linhas foram lidas e previstas ({'Portmap': 2702})
✔ Nenhum modelo foi treinado neste notebook (o treino está bloqueado e nenhuma tentativa foi feita) 
✔ Nenhum dos 672 arquivos que já existiam (pastas do SVM, do RF 07m/08m e do RF 05m/06m) foi alterado, sobrescrito ou apagado 
✔ Arquivos esperados gravados, para cada modelo 